In [1]:
import os
import numpy as np
import napari
from pathlib import Path
import zipfile
import gc

In [2]:
# mask‐file → short name mapping:
mask_files = {
    "results_perinuclear_mask.npz": "perinuclear",
    "results_transition_mask.npz": "transition",
    "results_telenuclear_mask.npz": "telenuclear",
}

sn = 13

base_dir = Path(os.getcwd()) / str(sn)

# --- extract raw.npz once ---
raw_npz = base_dir / "raw.npz"
raw_dir = base_dir / "raw_extracted_npz"
raw_npy = raw_dir / "arr_0.npy"

if not raw_npy.exists():
    raw_dir.mkdir(exist_ok=True)
    with np.load(raw_npz) as z:
        arr = z["arr_0"] if "arr_0" in z else list(z.values())[0]
    np.save(raw_npy, arr)

raw_data = np.load(raw_npy, mmap_mode="r")

# Extract results_binary_masks.npz once
binary_npz = base_dir / "results_binary_masks.npz"
binary_dir = base_dir / "binary_extracted_npz"
binary_npy = binary_dir / "arr_0.npy"

if not binary_npy.exists():
    binary_dir.mkdir(exist_ok=True)
    with np.load(binary_npz) as bi:
        arr = bi["arr_0"] if "arr_0" in bi else list(bi.values())[0]
    np.save(binary_npy, arr)

binary_data = np.load(binary_npy, mmap_mode='r')

# Free up memory used for extraction
try:
    del arr
except NameError:
    pass
    
gc.collect()

# folders = sorted(base_dir.glob(f"Sample{sn}_thresh_adj_mult_1.2_tele_overlap_{ov}"))
mask_layers = []

# --- loop through all overlap settings ---
for ov in np.linspace(0.60, 0.80, 5):
    ov_str = f"{ov:.2f}"
    subfolder = base_dir / f"Sample{sn}_thresh_adj_mult_1.2_tele_overlap_0.05_peri_overlap_{ov_str}"
    if not subfolder.exists():
        print(f"  [!] missing folder: {subfolder}")
        continue

    # extract each mask npz → .npy
    for fname, tag in mask_files.items():
        npz_path = subfolder / fname
        if not npz_path.exists():
            print(f"    [!] missing file: {npz_path.name}")
            continue

        ex_dir = subfolder / f"extracted_{tag}"
        ex_npy = ex_dir / f"{tag}.npy"

        if not ex_npy.exists():
            ex_dir.mkdir(exist_ok=True)
            with np.load(npz_path) as mz:
                m = mz["arr_0"] if "arr_0" in mz else list(mz.values())[0]
            np.save(ex_npy, m)

        # Free up memory used for extraction
        try:
            del m, mz
        except NameError:
            pass
        gc.collect()
        
        mask = np.load(ex_npy, mmap_mode="r")
        mask_layers.append((f'overlap_ratio_{ov}', mask))
        
with napari.gui_qt():
    viewer = napari.Viewer(title=f"Sample {sn}")
    viewer.add_image(
        raw_data,
        name=f"Sample{sn}_raw",
        blending="additive",
        colormap="gray")

    viewer.add_labels(binary_data, name=f'Sample{sn}_Perinucleus', opacity=0.5, visible=True)
    
    for name, mask in mask_layers:
        viewer.add_labels(mask, name=name, opacity=0.9, visible=False)

/home/volkan/micromamba/envs/filopodia_roi_selector/lib/python3.10/contextlib.py:135: FutureWarning: 
The 'gui_qt()' context manager is deprecated.
If you are running napari from a script, please use 'napari.run()' as follows:

    import napari

    viewer = napari.Viewer()  # no prior setup needed
    # other code using the viewer...
    napari.run()

In IPython or Jupyter, 'napari.run()' is not necessary. napari will automatically
start an interactive event loop for you: 

    import napari
    viewer = napari.Viewer()  # that's it!

  return next(self.gen)
/home/volkan/micromamba/envs/filopodia_roi_selector/lib/python3.10/site-packages/napari/_qt/qt_event_loop.py:338: FutureWarning: `QApplication` instance access through `get_app` is deprecated and will be removed in 0.6.0.
Please use `get_qapp` instead.

  app = get_app()
